# Crisis stress test
How would our portfolio have done in the three largest US equity crises since 2007?

Rule for this notebook: **only code patterns from the course notebooks** (`inputs/course/`).
The course source is named above every cell.

**The two portfolios we compare**

| Portfolio | What it holds |
|---|---|
| Strategy | 60% AGG (bonds) fixed + 40% satellite in SPY, GLD, DBC, each only while its trend is up; otherwise T-bills |
| 40/60 SPY/AGG | 40% stocks, 60% bonds, rebalanced monthly (our benchmark) |

Both hold 60% bonds. Strategy vs **40/60** shows what our diversified, trend-filtered 40% satellite
adds compared with simply holding 40% in SPY.

## Setup
**Course source:** Lecture 4, §4.0 (same imports).

**What happens here**
* `numpy`, `pandas`, `matplotlib`: the standard tools for numbers, tables and charts.
* `app`: our own project file `app.py`. It contains the full backtest (data download, signals,
  weights, costs). We do not rebuild it here; we only reuse its results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import app   # our backtest, same code as the submission notebook

## Step 1: monthly returns of the two portfolios
**Course source:** Lecture 3, Part 2 (`print(df.shape, df.index.min(), df.index.max())`).

**What happens here**
1. `app.download_data()` downloads the ETF prices and the T-bill yield from Yahoo Finance and
   turns them into month-end values.
2. `app.run_all(prices, rf)` runs the backtest. We only need `results`: a table with one row per
   month and one column per portfolio. Each number is that month's return **after costs**
   (0.02 = +2%).
3. The `print` line is the course's quick check: how many rows and columns, first and last date.
   We expect 236 months (Mar 2007 to Oct 2026) and 2 portfolios.
4. `results.head()` shows the first five rows so we can see what the data looks like.

In [ ]:
prices, rf = app.download_data()                         # download prices
results, rf_bt, weights, detail = app.run_all(prices, rf)  # run the backtest

print(results.shape, results.index.min(), results.index.max())  # quick check
results.head()                                                  # first rows

## Step 2: the three crisis windows
**Course source:** Lecture 2 (dictionaries), Lecture 4 (`df.loc['2024', ...]` to select dates).

**What happens here**
1. `crises` is a dictionary: each crisis name points to a (start, end) pair of months.
2. We chose the three largest falls of SPY at month-end since 2007. A window starts in the month
   **after** SPY's peak (the first month in which money was lost) and ends in the month of the
   lowest point:
   * Financial crisis: peak Oct 2007, low Feb 2009 → `'2007-11'` to `'2009-02'`
   * Covid crash: peak Dec 2019, low Mar 2020 → `'2020-01'` to `'2020-03'`
   * Inflation shock: peak Dec 2021, low Sep 2022 → `'2022-01'` to `'2022-09'`
3. `results.loc['2020-01':'2020-03']` cuts out all months between two dates (both included).
   We print one window as a check: three rows, January to March 2020.

In [ ]:
crises = {'Financial crisis': ('2007-11', '2009-02'),
          'Covid crash':      ('2020-01', '2020-03'),
          'Inflation shock':  ('2022-01', '2022-09')}

results.loc['2020-01':'2020-03']   # check one window

## Step 3: return and worst point in each crisis
**Course source:** Lecture 5 §5.5 (`(1 + r).prod()`), Lecture 4 §4.4 (`(1 + r).cumprod()`),
Lecture 4 (setting values with a condition, loops with `print`).

**What happens here, line by line**
1. `for name in crises:` repeats the block once per crisis.
2. `start, end = crises[name]` reads that crisis's two dates from the dictionary.
3. `r = results.loc[start:end]` keeps only the months of this crisis.
4. `(1 + r).prod() - 1` is the **total return over the crisis**: multiply all monthly growth
   factors (e.g. 1.02 × 0.95 × …), then subtract 1. Example: −0.20 means €100 became €80.
5. `(1 + r).cumprod()` is the **value path** of €1 invested at the start of the crisis, month by
   month. `.min()` is its lowest point; minus 1 gives the **worst point vs the start**.
6. `worst[worst > 0] = 0`: if a portfolio never fell below its starting value, its lowest point is
   above 1 and the formula would show a small gain. We set those cases to 0% (no loss).
7. The inner `for p in results.columns:` prints one line per portfolio. `:7.1%` formats the number
   as a percentage with one decimal.

**How to read it:** "return" is where the portfolio stood at the end of the crisis; "worst point"
is the deepest it went along the way (both measured from the start of the crisis).

In [ ]:
for name in crises:
    start, end = crises[name]
    r = results.loc[start:end]               # months of this crisis
    total = (1 + r).prod() - 1               # return over the whole crisis
    worst = (1 + r).cumprod().min() - 1      # lowest value vs the start
    worst[worst > 0] = 0                     # never below the start: no loss
    print(f"{name} ({start} to {end})")
    for p in results.columns:
        print(f"   {p:26} return {total[p]:7.1%}   worst point {worst[p]:7.1%}")

## Step 4: wealth during each crisis
**Course source:** Lecture 3 and 4 (`plt.subplots`, `.plot(ax=..., title=...)`, `plt.tight_layout()`).

**What happens here**
1. `plt.subplots(1, 3, figsize=(15, 4))` makes one row with three empty charts (`ax[0]`, `ax[1]`,
   `ax[2]`).
2. The loop takes one crisis at a time, cuts out its months as in Step 3, and draws
   `(1 + r).cumprod()`: the value of €1 invested at the start, for both portfolios.
3. `i` counts which of the three charts to draw into; `legend=(i == 0)` shows the legend only in
   the first chart to keep it clean.
4. `plt.tight_layout()` stops titles and labels from overlapping.

**How to read it:** a line at 0.90 means €1 had become €0.90. The first point is already after
the first crisis month, so lines do not start exactly at 1.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
i = 0
for name in crises:
    start, end = crises[name]
    r = results.loc[start:end]
    (1 + r).cumprod().plot(ax=ax[i], title=name, legend=(i == 0))  # value of 1 invested
    i = i + 1
plt.tight_layout()

**Limitation:** month-end data hides losses within a month. SPY fell about 34% from its
February 2020 high to the March low, but only 19% from month-end to month-end.